# 2.1 Limpieza de datos

Carga y exploración inicial del conjunto de datos del Titanic.

## Importar paquetes

En esta etapa utilizaremos únicamente pandas.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Carga de datos

In [2]:
df = pd.read_csv('./data/titanic.csv')

## Información básica del dataset

El método `sample(3)` selecciona aleatoriamente tres filas del DataFrame.

In [3]:
df.sample(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
727,728,1,3,"Mannion, Miss. Margareth",female,NaN,0,0,36866,7.7375,NaN,Q
842,843,1,1,"Serepeca, Miss. Augusta",female,30.0,0,0,113798,31.0000,NaN,C
792,793,0,3,"Sage, Miss. Stella Anna",female,NaN,8,2,CA. 2343,69.5500,NaN,S


Ahora revisamos las dimensiones del conjunto de datos.

In [4]:
print(f"El shape del conjunto de entrenamiento es {df.shape}. {df.shape[0]} filas y {df.shape[1]} columnas.")

El shape del conjunto de entrenamiento es (891, 12). 891 filas y 12 columnas.


## Estructura y tipos de datos

Usamos `info()` para revisar las columnas, los valores no nulos, los tipos de datos y el uso de memoria.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


### Observaciones

El conjunto de datos contiene **7 variables numéricas**:

- Variables decimales (`float`): `Age` y `Fare`.
- Variables enteras (`int`): `PassengerId`, `Survived`, `Pclass`, `SibSp` y `Parch`.

También contiene **5 variables categóricas o de texto**:

- `Name`
- `Sex`
- `Ticket`
- `Cabin`
- `Embarked`

> En pandas 3 estas columnas aparecen con el tipo `str`; en versiones anteriores suelen aparecer como `object`. En ambos casos representan texto y requieren un tratamiento categórico para el modelado.

Finalmente, se identifican valores faltantes en `Age`, `Cabin` y `Embarked`. Estas columnas deberán analizarse antes de decidir si conviene imputar sus valores, transformarlas o eliminarlas.

## Corrección de valores faltantes

### Columna `Cabin`

Como `Cabin` tiene 687 valores faltantes (77.10 % del conjunto), eliminaremos la columna del DataFrame.

In [6]:
df.drop('Cabin', axis=1, inplace=True)

In [7]:
print(f"Dimensiones después de eliminar Cabin: {df.shape}")
df.columns

Dimensiones después de eliminar Cabin: (891, 11)


Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Embarked'],
      dtype='str')

### Columnas `Age` y `Embarked`

Conservaremos `Age` e imputaremos sus valores faltantes con la **media**. Para `Embarked`, una variable categórica con solo dos datos faltantes, utilizaremos la **moda**.

In [8]:
df['Age'] = df['Age'].fillna(df['Age'].mean())

In [9]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

### Verificación de valores faltantes

In [10]:
round(df.isnull().sum().sort_values(ascending=False) / len(df) * 100, 2)

PassengerId    0.0
Survived       0.0
Pclass         0.0
Name           0.0
Sex            0.0
Age            0.0
SibSp          0.0
Parch          0.0
Ticket         0.0
Fare           0.0
Embarked       0.0
dtype: float64

In [11]:
df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

## Valores duplicados

Verificamos si el conjunto contiene filas repetidas.

In [12]:
df.duplicated().sum()

np.int64(0)

## Eliminación de columnas poco útiles

Eliminaremos las siguientes columnas en su formato original:

- **`PassengerId`**: es un identificador único y no describe características generalizables del pasajero.
- **`Name`**: el nombre completo no es directamente útil. Sus títulos podrían aprovecharse mediante ingeniería de características, pero no se utilizarán en esta versión.
- **`Ticket`**: sus valores son únicos o casi únicos y su formato bruto no aporta información estructurada directamente aprovechable.

In [13]:
df.drop('Name', axis=1, inplace=True)
df.drop('Ticket', axis=1, inplace=True)
df.drop('PassengerId', axis=1, inplace=True)

In [14]:
print(f"Después de limpiar df: {df.shape[0]} filas y {df.shape[1]} columnas.")

Después de limpiar df: 891 filas y 8 columnas.


## Guardar datos limpios

Guardamos un checkpoint para que los siguientes notebooks trabajen directamente con los datos ya limpiados.

In [15]:
df.to_csv('./data/titanic_clean.csv', index=False)